# Historical generation and demand audit

This notebook describes the locally built FY2020–FY2025 regional generation-demand panel. Price analysis is deliberately deferred until the AEMO `DISPATCHPRICE` archive is acquired and joined; do not infer price effects from this notebook.

In [1]:
from pathlib import Path
import pandas as pd

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
panel = pd.read_parquet(root / 'data/processed/nem_region_hour_generation_demand.parquet')
panel.shape, panel[['timestamp', 'region']].agg(['min', 'max'])

((263040, 28),
                     timestamp region
 min 2019-07-01 00:00:00+10:00   NSW1
 max 2025-06-30 23:00:00+10:00   VIC1)

In [2]:
summary = panel.groupby('region', observed=True)[['demand_mw', 'wind_mw', 'solar_utility_mw', 'hydro_mw', 'renewable_share_ws', 'renewable_share_broad']].agg(['mean', 'median', 'std'])
summary.round(3)

demand_mw                       wind_mw                    \
            mean    median       std      mean   median      std   
region                                                             
NSW1    7587.325  7450.350  1313.196   654.208  581.016  429.568   
QLD1    6114.560  5989.383  1000.667   240.354  197.900  185.956   
SA1     1273.817  1301.275   398.527   704.711  643.374  460.640   
TAS1    1128.242  1107.634   146.974   193.636  170.178  136.483   
VIC1    4766.149  4650.194   967.143  1040.948  888.191  776.815   

       solar_utility_mw                  hydro_mw                    \
                   mean  median      std     mean   median      std   
region                                                                
NSW1            552.186  22.665  785.518  321.315  183.947  363.137   
QLD1            516.901  27.725  670.068  127.216   86.422  117.645   
SA1              86.119   1.835  122.462    0.000    0.000    0.000   
TAS1              0.000   0.000    0.000  945.491  878.824  422.934   
VIC1            163.094   6.558  224.876  300.378  144.771  369.905   

       renewable_share_ws               renewable_share_broad                
                     mean median    std                  mean median    std  
region                                                                       
NSW1                0.169  0.131  0.131                 0.208  0.178  0.123  
QLD1                0.133  0.083  0.124                 0.152  0.111  0.120  
SA1                 0.711  0.649  1.802                 0.711  0.649  1.802  
TAS1                0.176  0.153  0.127                 0.999  0.953  0.305  
VIC1                0.264  0.230  0.176                 0.320  0.295  0.158

In [3]:
monthly = (panel.assign(month=panel.timestamp.dt.strftime('%Y-%m'))
    .groupby(['month', 'region'], observed=True)['renewable_share_ws'].mean()
    .unstack('region'))
monthly.tail(12).round(3)

region,NSW1,QLD1,SA1,TAS1,VIC1
month,,,,,
2024-07,0.189,0.177,0.665,0.149,0.272
2024-08,0.205,0.181,0.855,0.223,0.374
2024-09,0.272,0.200,1.014,0.294,0.417
2024-10,0.271,0.217,1.690,0.240,0.389
2024-11,0.270,0.211,0.981,0.187,0.311
2024-12,0.294,0.177,0.964,0.244,0.359
2025-01,0.314,0.190,0.744,0.171,0.362
2025-02,0.277,0.196,0.700,0.203,0.419
2025-03,0.265,0.172,0.745,0.154,0.350
